# Expense Category Classification Notebook

This notebook builds and compares category prediction approaches for personal finance transactions.

In [1]:
!pip install pandas scikit-learn datasets scipy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip


## 1. Environment Setup

Install required libraries for data processing and modeling.

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Data Loading and Inspection

Load the dataset and inspect its structure.

In [3]:
url = "https://huggingface.co/buckets/Diamond-12324/Expense-Tracker/resolve/personal_expense_classification%20-%20Final.csv?download=true"

df = pd.read_csv(url, encoding="utf-8")

df.head()

,Date,Transaction Description,Category,Amount,Type,Year,Month,Day,Day_of_Week,Is_Weekend,Quarter
0,2020-01-01,Game purchase,Entertainment,237.87,Expense,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01,Gift received,Other Income,1791.21,Income,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01,Tuition fee,Education,241.05,Expense,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01,Insurance payment,Healthcare,130.88,Expense,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01,Bought clothes,Shopping,81.69,Expense,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
print(df.shape)
print(df.columns)
df["Category"].value_counts()

(51482, 11)
Index(['Date', 'Transaction Description', 'Category', 'Amount', 'Type', 'Year',
       'Month', 'Day', 'Day_of_Week', 'Is_Weekend', 'Quarter'],
      dtype='str')


Category
Healthcare       3760
Shopping         3760
Transport        3730
Food             3726
Utilities        3561
Education        3504
Misc             3503
Housing          3499
Entertainment    3463
Childcare        3447
Investment       3212
Freelance        3149
Other Income     3084
Salary           3069
Rental           3015
Name: count, dtype: int64

## 3. Feature Engineering

Clean text, extract date-based features, and encode target labels.

In [5]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\b(gasoline|petrol|fuel|diesel)\b", "petrol", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.strip()

df["text"] = df["Transaction Description"].apply(clean_text)

In [6]:
df["Date"] = pd.to_datetime(df["Date"])

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["Day_of_Week"] = df["Date"].dt.dayofweek
df["Is_Weekend"] = df["Day_of_Week"].isin([5, 6]).astype(int)
df["Quarter"] = df["Date"].dt.quarter

In [7]:
le = LabelEncoder()
df["y"] = le.fit_transform(df["Category"])

In [8]:
df["Type"] = df["Type"].map({
    "Expense": 0,
    "Income": 1
})

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    df, df["y"], test_size=0.2, random_state=42
)

## 4. Model 1: Keyword Rule-Based

Create simple keyword rules and evaluate baseline accuracy.

In [10]:
keyword_rules = {
    "Entertainment": ["game", "movie", "netflix"],
    "Education": ["tuition", "course", "school"],
    "Healthcare": ["insurance", "hospital", "medicine"],
    "Shopping": ["clothes", "shopping", "buy"],
}

In [11]:
def keyword_predict(text):
    for category, keywords in keyword_rules.items():
        for kw in keywords:
            if kw in text:
                return category
    return "Unknown"

In [12]:
y_pred_kw = [keyword_predict(x) for x in X_test["text"]]

y_pred_kw_encoded = [
    le.transform([y])[0] if y in le.classes_ else -1
    for y in y_pred_kw
]

valid_idx = [i for i, y in enumerate(y_pred_kw_encoded) if y != -1]

acc_kw = accuracy_score(
    [y_test.iloc[i] for i in valid_idx],
    [y_pred_kw_encoded[i] for i in valid_idx]
)

print("Keyword Accuracy:", acc_kw)

Keyword Accuracy: 0.7871287128712872


## 5. Model 2: TF-IDF + Structured Features

Vectorize transaction text with TF-IDF, combine with numeric features, and train Logistic Regression.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_text = vectorizer.fit_transform(X_train["text"])
X_test_text = vectorizer.transform(X_test["text"])

In [14]:
num_features = ["Amount", "Type", "Month", "Day_of_Week", "Is_Weekend"]

# Fill missing values
X_train_num = X_train[num_features].copy()
X_test_num = X_test[num_features].copy()

X_train_num = X_train_num.fillna(0)
X_test_num = X_test_num.fillna(0)

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_num = scaler.fit_transform(X_train_num)
X_test_num = scaler.transform(X_test_num)

In [16]:
from scipy.sparse import hstack

X_train_combined = hstack([X_train_text, X_train_num])
X_test_combined = hstack([X_test_text, X_test_num])

In [17]:
from sklearn.linear_model import LogisticRegression

tfidf_model = LogisticRegression(max_iter=1000)
tfidf_model.fit(X_train_combined, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [18]:
y_pred_tfidf = tfidf_model.predict(X_test_combined)

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)

print("TF-IDF + Structured Accuracy:", acc_tfidf)

TF-IDF + Structured Accuracy: 0.9986403806934059


## 6. Model 3 (Optional, Run At End): MiniLM + Structured Features

Run this section at the end of the notebook if you want to compare against a transformer embedding model. You will be asked for confirmation before running it, and `sentence-transformers` will be installed only if you approve.

In [19]:
import importlib.util
import subprocess
import sys

run_minilm = input("Run optional MiniLM model now? (y/n): ").strip().lower() in {"y", "yes"}

if run_minilm:
    if importlib.util.find_spec("sentence_transformers") is None:
        print("Installing sentence-transformers...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer("paraphrase-MiniLM-L3-v2")
else:
    print("Skipping MiniLM model by user choice.")

Skipping MiniLM model by user choice.


In [20]:
if run_minilm:
    X_train_emb = embedder.encode(X_train["text"].tolist())
    X_test_emb = embedder.encode(X_test["text"].tolist())
else:
    X_train_emb = None
    X_test_emb = None

In [21]:
if run_minilm:
    X_train_final = np.hstack([X_train_emb, X_train_num])
    X_test_final = np.hstack([X_test_emb, X_test_num])
else:
    X_train_final = None
    X_test_final = None

In [22]:
if run_minilm:
    mini_model = LogisticRegression(max_iter=1000)
    mini_model.fit(X_train_final, y_train)
else:
    mini_model = None

In [23]:
if run_minilm:
    y_pred_mini = mini_model.predict(X_test_final)

    acc_mini = accuracy_score(y_test, y_pred_mini)

    print("MiniLM + Structured Accuracy:", acc_mini)
    print(classification_report(y_test, y_pred_mini, target_names=le.classes_))
else:
    acc_mini = np.nan
    print("MiniLM model skipped by user choice.")

MiniLM model skipped by user choice.


## 7. Model Comparison (Optional)

Compare all three model accuracies. If you skipped Model 3, ignore its result row.

In [24]:
results = pd.DataFrame({
    "Model": [
        "Keyword-Based",
        "TF-IDF + Structured",
        "MiniLM + Structured"
    ],
    "Accuracy": [
        acc_kw,
        acc_tfidf,
        acc_mini
    ]
})

results

,Model,Accuracy
0,Keyword-Based,0.787129
1,TF-IDF + Structured,0.998640
2,MiniLM + Structured,NaN


## 8. Quick User Input Test

Use this helper function to test a single transaction input with a selected model (`keyword`, `tfidf`, or optional `minilm`).

In [25]:
def predict_transaction_category(
    description,
    amount=0.0,
    transaction_type="Expense",
    date_str="2026-01-01",
    model_name="tfidf",
):
    """Predict a category for one transaction using keyword, tfidf, or optional minilm."""
    model_name = model_name.lower().strip()

    cleaned = clean_text(description)
    parsed_date = pd.to_datetime(date_str)

    type_value = 1 if str(transaction_type).lower() == "income" else 0
    day_of_week = int(parsed_date.dayofweek)
    is_weekend = 1 if day_of_week in [5, 6] else 0

    input_num = pd.DataFrame(
        [{
            "Amount": float(amount),
            "Type": type_value,
            "Month": int(parsed_date.month),
            "Day_of_Week": day_of_week,
            "Is_Weekend": is_weekend,
        }]
    )
    input_num_scaled = scaler.transform(input_num)

    if model_name == "keyword":
        pred_label = keyword_predict(cleaned)
        return pred_label

    if model_name == "tfidf":
        input_text = vectorizer.transform([cleaned])
        input_features = hstack([input_text, input_num_scaled])
        pred_idx = tfidf_model.predict(input_features)[0]
        return le.inverse_transform([pred_idx])[0]

    if model_name == "minilm":
        if (
            "run_minilm" not in globals()
            or not run_minilm
            or "embedder" not in globals()
            or "mini_model" not in globals()
            or mini_model is None
        ):
            return "MiniLM model is optional and not loaded. Confirm and run Section 6 first."
        input_emb = embedder.encode([cleaned])
        input_features = np.hstack([input_emb, input_num_scaled])
        pred_idx = mini_model.predict(input_features)[0]
        return le.inverse_transform([pred_idx])[0]

    return "Invalid model_name. Use 'keyword', 'tfidf', or 'minilm'."


# Quick interactive test
user_desc = input("Enter transaction description: ")
user_amt = float(input("Enter amount (e.g., 15.5): "))
user_type = input("Enter type (Expense/Income): ")
user_date = input("Enter date (YYYY-MM-DD): ")
user_model = input("Choose model (keyword/tfidf/minilm): ")

print(
    "Predicted category:",
    predict_transaction_category(
        description=user_desc,
        amount=user_amt,
        transaction_type=user_type,
        date_str=user_date,
        model_name=user_model,
    ),
)

Predicted category: Transport


In [28]:
examples = [
    {
        "description": "Bought rice and vegetables at Psah Thmei",
        "amount": 4.5,
        "transaction_type": "Expense",
        "date_str": "2026-04-03",
        "category": "Groceries",
    },
    {
        "description": "Paid for gasoline at PTT station",
        "amount": 6,
        "transaction_type": "Expense",
        "date_str": "2026-04-07",
        "category": "Transportation",
    },
    {
        "description": "Grabbed Bai Sach Chrouk breakfast near house",
        "amount": 1.5,
        "transaction_type": "Expense",
        "date_str": "2026-04-12",
        "category": "Food & Dining",
    },
    {
        "description": "Bought drinking water 20L jug",
        "amount": 2,
        "transaction_type": "Expense",
        "date_str": "2026-04-15",
        "category": "Groceries",
    },
    {
        "description": "Topped up Cellcard mobile data",
        "amount": 5,
        "transaction_type": "Expense",
        "date_str": "2026-04-18",
        "category": "Utilities",
    },
    {
        "description": "Paid tuk-tuk fare to Orussey Market",
        "amount": 3,
        "transaction_type": "Expense",
        "date_str": "2026-04-09",
        "category": "Transportation",
    },
    {
        "description": "Bought Nom Pang sandwich from street stall",
        "amount": 1,
        "transaction_type": "Expense",
        "date_str": "2026-04-21",
        "category": "Food & Dining",
    },
    {
        "description": "Paid electricity bill at EDC office",
        "amount": 25,
        "transaction_type": "Expense",
        "date_str": "2026-04-24",
        "category": "Utilities",
    },
    {
        "description": "Purchased snacks and drinks at Lucky Supermarket",
        "amount": 8,
        "transaction_type": "Expense",
        "date_str": "2026-04-17",
        "category": "Groceries",
    },
    {
        "description": "Refilled motorbike gasoline at Caltex",
        "amount": 4,
        "transaction_type": "Expense",
        "date_str": "2026-04-28",
        "category": "Transportation",
    },
]

In [30]:
from pathlib import Path
import joblib
import pandas as pd
from scipy.sparse import hstack

artifact_path = Path("../../models/tfidf_expense_classifier.joblib")
artifact_path.parent.mkdir(parents=True, exist_ok=True)

# Set this to True only when you intentionally want to replace the saved file.
overwrite_existing = False

required_train_objects = ["vectorizer", "scaler", "tfidf_model", "le", "num_features", "clean_text"]
can_export = all(name in globals() for name in required_train_objects)

if can_export:
    artifact = {
        "vectorizer": vectorizer,
        "scaler": scaler,
        "model": tfidf_model,
        "label_encoder": le,
        "num_features": num_features,
        "clean_text": clean_text,
    }

    if artifact_path.exists() and not overwrite_existing:
        print(f"Model already exists, keeping current file: {artifact_path.resolve()}")
    else:
        joblib.dump(artifact, artifact_path)
        print(f"Saved TF-IDF model artifact to: {artifact_path.resolve()}")
else:
    print("Training objects not found in memory. Skipping export.")

if artifact_path.exists():
    loaded = joblib.load(artifact_path)

    def predict_from_exported_model(
        description,
        amount=0.0,
        transaction_type="Expense",
        date_str="2026-01-01",
    ):
        cleaner = loaded["clean_text"]
        vect = loaded["vectorizer"]
        sc = loaded["scaler"]
        mdl = loaded["model"]
        enc = loaded["label_encoder"]

        cleaned = cleaner(description)
        parsed_date = pd.to_datetime(date_str)

        row = pd.DataFrame([
            {
                "Amount": float(amount),
                "Type": 1 if str(transaction_type).lower() == "income" else 0,
                "Month": int(parsed_date.month),
                "Day_of_Week": int(parsed_date.dayofweek),
                "Is_Weekend": 1 if int(parsed_date.dayofweek) in [5, 6] else 0,
            }
        ])

        text_features = vect.transform([cleaned])
        num_features_scaled = sc.transform(row)
        features = hstack([text_features, num_features_scaled])

        pred_idx = mdl.predict(features)[0]
        return enc.inverse_transform([pred_idx])[0]

    print(
        "Sample prediction from exported model:",
        predict_from_exported_model(
            description="Paid for groceries at local market",
            amount=23.5,
            transaction_type="Expense",
            date_str="2026-04-06",
        ),
    )

    print(
        "Sample prediction from exported model:",
        predict_from_exported_model(
            description="Paid for Gasoline at Tela",
            amount=3,
            transaction_type="Expense",
            date_str="2026-04-06",
        ),
    )

    for item in examples:
        predict_from_exported_model(
            description=item["description"],
            amount=item["amount"],
            transaction_type=item["transaction_type"],
            date_str=item["date_str"],
        )

        print(
            f"Description: {item['description']}\n"
            f"Actual Category: {item['category']}\n"
            f"Predicted Category: {predict_from_exported_model(item['description'], item['amount'], item['transaction_type'], item['date_str'])}\n"
        )

else:
    print(f"No exported model found at: {artifact_path.resolve()}")

Model already exists, keeping current file: /workspaces/AI-Budget-Model/models/tfidf_expense_classifier.joblib
Sample prediction from exported model: Food
Sample prediction from exported model: Transport
Description: Bought rice and vegetables at Psah Thmei
Actual Category: Groceries
Predicted Category: Food

Description: Paid for gasoline at PTT station
Actual Category: Transportation
Predicted Category: Transport

Description: Grabbed Bai Sach Chrouk breakfast near house
Actual Category: Food & Dining
Predicted Category: Food

Description: Bought drinking water 20L jug
Actual Category: Groceries
Predicted Category: Shopping

Description: Topped up Cellcard mobile data
Actual Category: Utilities
Predicted Category: Utilities

Description: Paid tuk-tuk fare to Orussey Market
Actual Category: Transportation
Predicted Category: Transport

Description: Bought Nom Pang sandwich from street stall
Actual Category: Food & Dining
Predicted Category: Food

Description: Paid electricity bill at 

In [33]:
transactions = [
    (
        "Paid for Gasoline at Tela",
        predict_from_exported_model(
            description="Paid for Gasoline at Tela",
            amount=3,
            transaction_type="Expense",
            date_str="2026-04-06",
        ),
        "Transportation"
    ),
    (
        "Bought iced coffee at Brown Coffee",
        predict_from_exported_model(
            description="Bought iced coffee at Brown Coffee",
            amount=2.5,
            transaction_type="Expense",
            date_str="2026-04-12",
        ),
        "Food & Drink"
    ),
    (
        "Paid for breakfast noodles at local shop",
        predict_from_exported_model(
            description="Paid for breakfast noodles at local shop",
            amount=1.75,
            transaction_type="Expense",
            date_str="2026-04-03",
        ),
        "Food & Drink"
    ),
    (
        "Bought lunch rice and pork at market",
        predict_from_exported_model(
            description="Bought lunch rice and pork at market",
            amount=2.25,
            transaction_type="Expense",
            date_str="2026-04-18",
        ),
        "Food & Drink"
    ),
    (
        "Paid for tuk tuk ride to Aeon Mall",
        predict_from_exported_model(
            description="Paid for tuk tuk ride to Aeon Mall",
            amount=4,
            transaction_type="Expense",
            date_str="2026-04-09",
        ),
        "Transportation"
    ),
    (
        "Bought snacks and water at 7-Eleven",
        predict_from_exported_model(
            description="Bought snacks and water at 7-Eleven",
            amount=2,
            transaction_type="Expense",
            date_str="2026-04-21",
        ),
        "Groceries"
    ),
    (
        "Paid for phone top up Smart 5USD",
        predict_from_exported_model(
            description="Paid for phone top up Smart 5USD",
            amount=5,
            transaction_type="Expense",
            date_str="2026-04-15",
        ),
        "Utilities"
    ),
    (
        "Bought vegetables and fish at Psar Thmei market",
        predict_from_exported_model(
            description="Bought vegetables and fish at Psar Thmei market",
            amount=6,
            transaction_type="Expense",
            date_str="2026-04-26",
        ),
        "Groceries"
    ),
    (
        "Paid for milk tea at Chatime",
        predict_from_exported_model(
            description="Paid for milk tea at Chatime",
            amount=2.75,
            transaction_type="Expense",
            date_str="2026-04-07",
        ),
        "Food & Drink"
    ),
    (
        "Paid for laundry service near home",
        predict_from_exported_model(
            description="Paid for laundry service near home",
            amount=3.5,
            transaction_type="Expense",
            date_str="2026-04-29",
        ),
        "Personal Care"
    ),
]

print("Predicted categories for sample transactions:\n")

for desc, pred, actual in transactions:
    print(f"Description: {desc}")
    print(f"Predicted: {pred} | Actual: {actual}")
    print("-" * 50)

Predicted categories for sample transactions:

Description: Paid for Gasoline at Tela
Predicted: Transport | Actual: Transportation
--------------------------------------------------
Description: Bought iced coffee at Brown Coffee
Predicted: Food | Actual: Food & Drink
--------------------------------------------------
Description: Paid for breakfast noodles at local shop
Predicted: Food | Actual: Food & Drink
--------------------------------------------------
Description: Bought lunch rice and pork at market
Predicted: Food | Actual: Food & Drink
--------------------------------------------------
Description: Paid for tuk tuk ride to Aeon Mall
Predicted: Transport | Actual: Transportation
--------------------------------------------------
Description: Bought snacks and water at 7-Eleven
Predicted: Food | Actual: Groceries
--------------------------------------------------
Description: Paid for phone top up Smart 5USD
Predicted: Shopping | Actual: Utilities
----------------------------